# HPC Cluster Concepts

## Cluster Overview

<img src="./img/general_cluster_architecture.png"/>

A **cluster** is a collection of compute nodes. A **partition** is a logical grouping of nodes or resources. It is used to organize access to different types of hardware, such as CPU nodes, GPU nodes, or nodes with special characteristics.

A **node** is one compute server. Each node can contain one or more CPU sockets. A socket is the physical CPU package installed in the server.

Each socket contains multiple **CPU cores**. A CPU core is a physical computing unit. Depending on the processor and configuration, each core may expose one or more **CPU hardware threads**, typically two when simultaneous multithreading or hyperthreading is enabled. If this is disabled, one hardware thread usually corresponds to one physical core.

Modern processors are often divided into **NUMA domains**, also called locality domains. A NUMA domain is a group of CPUs that share memory that is physically closer to them. Accessing local memory is faster than accessing memory attached to another NUMA domain. A single socket can therefore contain one or more NUMA domains, depending on the processor architecture and BIOS configuration.

Some nodes may also contain **GPUs**, which are accelerator processors. GPUs can have locality relationships with specific CPU sockets or NUMA domains, so their placement in the hardware topology can matter for performance.

A **task** is a running process, for example an *MPI rank*. **Affinity** describes where that process is allowed to run: on which CPU, core, socket, or NUMA domain. Correct affinity helps keep processes close to the CPU cores, memory, and accelerators they use.

A simplified hierarchy is:

```shell
Cluster
  └── Partition
       └── Node
            └── Socket
                 └── NUMA / Locality Domain
                      └── CPU Core
                           └── CPU Hardware Thread
```

The key message is that performance is not only about how many CPUs a system has, but also about **where those CPUs, memory, tasks, and accelerators are located relative to each other.**

## Slurm Cluster Overview

<img src="./img/general_slurm_architecture.png"/>

**Slurm** is the workload management layer that connects users to the compute infrastructure. In other words, **the HPC cluster provides the resources, while Slurm manages access to them**. It queues jobs, allocates resources, launches workloads on compute nodes, and records usage for accounting and reporting.

An **HPC cluster** provides the physical and shared resources: login nodes, compute nodes, CPUs, memory, GPUs, high-speed network, and often **shared storage**. Users normally access the cluster through login nodes or other entry points, but heavy workloads should not run directly there. Instead, users submit jobs to Slurm using commands such as `sbatch` or `srun`.

Slurm receives these job requests, places them in a queue, and decides when and where they can run based on available resources, partitions, priorities, limits, and site policies. The Slurm controller, **slurmctld**, is responsible for scheduling and launching jobs on the selected compute nodes.

Once resources are available, Slurm starts the job on the compute nodes allocated to it. These nodes provide the CPUs, memory, and GPUs requested by the job.

Slurm also optionally includes an accounting component called **slurmdbd**. The **slurmdbd** service collects and stores job accounting information in a database. This includes details such as who ran a job, when it started and ended, which resources were allocated, how long the job ran, and how much CPU, memory, or GPU time was consumed. This information is used for reporting, fair-share calculations, usage analysis, debugging, and long-term accounting.

### The General HPC Slurm cluster

At **Paul Scherrer Institut**, **Merlin** is a series of centrally managed high-performance computing clusters, and is providing PSI staff and collaborators with scalable CPU and GPU resources for simulations, data analysis, and other data-intensive scientific workloads. This is operated by the [HPCE team](https://www.psi.ch/en/awi/hpce-group), in the [Scientific Computing Division](https://www.psi.ch/en/csd). PSI also run other Slurm clusters

**Merlin7** is the newest generation of the Merlin HPC environment. The cluster is based on Slurm, and is hosted on CSCS's Alps infrastructure in Lugano as an independent vCluster, and provides PSI users with flexible access to CPU and modern GPU resources for demanding scientific computing workloads.

In addition to Merlin, PSI also operates more specialized computing clusters for specific facilities, projects, or departments, such as the **Ra** data analysis cluster and the **SwissFEL** online/near-time computing infrastructure, which are tailored to experimental data acquisition, processing, and analysis workflows.

### Accessing the HPC cluster

Users normally access **Merlin7** through **login nodes** or through web-based entry services. These systems are the entry point to the cluster, but they are not intended for heavy computation.

For Merlin7, the main SSH login nodes are:

* `login001.merlin7.psi.ch`
* `login002.merlin7.psi.ch`

Users can connect to either login node using an SSH client:

```bash
ssh $USER@login001.merlin7.psi.ch
ssh $USER@login002.merlin7.psi.ch
```

The login nodes are used for tasks such as editing files, preparing job scripts, compiling software, checking job status, and submitting jobs to the batch system. Long-running or resource-intensive workloads should be submitted to the scheduler instead of being executed directly on the login nodes.

Users without a local SSH client can also access the cluster through browser-based services, for example:
* https://ondemand.psi.ch
* https://merlin7-nx-01.psi.ch:4443/

These services provide alternative entry points, such as web terminals, graphical desktops, Jupyter sessions, or application portals, depending on the service configuration.

## Hands-on: understanding hardware topology, NUMA, cores, threads, and affinity

### Duration
Around 15 minutes.

### Requirements

Useful tools:

```bash
lscpu
numactl
hwloc-ls
lstopo
taskset
ps
top
```

### Exercises

#### Exercise 1: Access the Merlin7 cluster

#### Exercise 2: Identify the machine topology

Understand the basic hardware hierarchy of the nodes.

On the login node, run:

```bash
lscpu
```

Then focus on the most relevant fields:

```bash
lscpu | egrep 'CPU\(s\)|Thread|Core|Socket|NUMA|Model name'
```

Now run the same command on a compute node, as follows:

```bash
srun -p interactive --reservation interactive \
  bash -lc "lscpu | egrep 'CPU\(s\)|Thread|Core|Socket|NUMA|Model name'"
```



##### Questions

> **[INFO]** Edit this cell and write your answers below.

1. **How many sockets do the nodes have?**
> YOUR ANSWER HERE

3. **How many physical cores per socket?**
> YOUR ANSWER HERE

4. **How many hardware threads per core?**
> YOUR ANSWER HERE

5. **How many total hardware threads?**
> YOUR ANSWER HERE

6. **How many NUMA domains? Why do they differ if the processor is the same?**
> YOUR ANSWER HERE
